In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class CandleDataset(Dataset):
    def __init__(self, df, feature_cols, window=60):
        self.features = df[feature_cols].values.astype(np.float32)
        self.labels = df["label"].values.astype(np.int64)
        self.window = window

    def __len__(self):
        return len(self.features) - self.window

    def __getitem__(self, idx):
        x = self.features[idx : idx + self.window]          # (window, n_features)
        y = self.labels[idx + self.window - 1]                # лейбл на конец окна
        return torch.tensor(x), torch.tensor(y)

In [ ]:
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score
from model import LSTMClassifier
import torch.nn as nn

def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3, device="cuda"):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # валидация
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                logits = model(x)
                preds = logits.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(y.numpy())

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average="macro")
        print(f"Epoch {epoch+1}: train_loss={train_loss/len(train_loader):.4f}, val_acc={acc:.4f}, val_f1={f1:.4f}")

    return model

In [ ]:
import pandas as pd
from model import prepare_features_and_labels, feature_cols

# load train data
df = pd.read_parquet("data/synthetic/train/mins_1440_seed487_bodysigma0046_wickdf23.parquet")
df = prepare_features_and_labels(df, horizon=5, flat_threshold=0.001)

# train/val split
split_idx = int(len(df) * 0.8)
train_df, val_df = df.iloc[:split_idx], df.iloc[split_idx:]

train_ds = CandleDataset(train_df, feature_cols, window=60)
val_ds = CandleDataset(val_df, feature_cols, window=60)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)   # shuffle only batches
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

model = LSTMClassifier(n_features=len(feature_cols), num_classes=3)
model = train_model(model, train_loader, val_loader, epochs=200, device="cuda" if torch.cuda.is_available() else "cpu")

Epoch 1: train_loss=1.0043, val_acc=0.5576, val_f1=0.2387
Epoch 2: train_loss=0.8052, val_acc=0.5576, val_f1=0.2387
Epoch 3: train_loss=0.7654, val_acc=0.5576, val_f1=0.2387
Epoch 4: train_loss=0.7580, val_acc=0.5576, val_f1=0.2387
Epoch 5: train_loss=0.7494, val_acc=0.5576, val_f1=0.2387
Epoch 6: train_loss=0.7331, val_acc=0.5576, val_f1=0.2387
Epoch 7: train_loss=0.7512, val_acc=0.5576, val_f1=0.2387
Epoch 8: train_loss=0.7378, val_acc=0.5576, val_f1=0.2387
Epoch 9: train_loss=0.7341, val_acc=0.5576, val_f1=0.2387
Epoch 10: train_loss=0.7328, val_acc=0.5576, val_f1=0.2387
Epoch 11: train_loss=0.7341, val_acc=0.5576, val_f1=0.2387
Epoch 12: train_loss=0.7532, val_acc=0.5576, val_f1=0.2387
Epoch 13: train_loss=0.7300, val_acc=0.5576, val_f1=0.2387
Epoch 14: train_loss=0.7336, val_acc=0.5576, val_f1=0.2387
Epoch 15: train_loss=0.7412, val_acc=0.5576, val_f1=0.2387
Epoch 16: train_loss=0.7332, val_acc=0.5576, val_f1=0.2387
Epoch 17: train_loss=0.7336, val_acc=0.5576, val_f1=0.2387
Epoch 

In [33]:
torch.save(model.state_dict(), "models/lstm_0046_23.pt")